# Домашнее задание: cравнение древовидных алгоритмов на NLP-данных

**Задача:** oбучить разные древовидные модели на текстах новостей, замерить test accuracy и время обучения, заполнить таблицы.

## 1. Загрузка и подготовка данных (код дан)

In [9]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00


In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time
import pandas as pd

# Загружаем 3 категории
categories = ['rec.sport.baseball', 'sci.space', 'comp.graphics']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

# TF-IDF векторизация
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X = vectorizer.fit_transform(newsgroups.data)
y = newsgroups.target

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Размер обучающей выборки: (2363, 5000)
Размер тестовой выборки: (591, 5000)


## 2. Образец: Дерево решений (Decision Tree)

**Гиперпараметр:** `max_depth`

In [2]:
from sklearn.tree import DecisionTreeClassifier

results_tree = []

# Образец: мы делаем перебор значений max_depth = [3, 5, 10, 20, None])
# Для каждого: замер времени, обучение, accuracy на тесте, сохранение в results_tree

for depth in [3, 5, 10, 20, None]:
    start = time.time()
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_tree.append({'max_depth': depth, 'test_accuracy': round(acc, 4), 'time_sec': round(fit_time, 2)})
    print(f"depth={depth}: acc={acc:.4f}, time={fit_time:.2f}s")

df_tree = pd.DataFrame(results_tree)
print("\nТаблица 1. Дерево решений")
print(df_tree.to_markdown(index=False))

depth=3: acc=0.6244, time=0.08s
depth=5: acc=0.6971, time=0.14s
depth=10: acc=0.7716, time=0.25s
depth=20: acc=0.8409, time=0.33s
depth=None: acc=0.8697, time=0.40s

Таблица 1. Дерево решений
|   max_depth |   test_accuracy |   time_sec |
|------------:|----------------:|-----------:|
|           3 |          0.6244 |       0.08 |
|           5 |          0.6971 |       0.14 |
|          10 |          0.7716 |       0.25 |
|          20 |          0.8409 |       0.33 |
|         nan |          0.8697 |       0.4  |


## 3. Заполните таблицы для остальных алгоритмов

По аналогии с образцом обучите следующие модели и запишите результаты.

### 3.1. Случайный лес (Random Forest)

**Гиперпараметр:** `max_depth` (фиксируем `n_estimators=100`)

| max_depth | test_accuracy | time_sec |
|-----------|---------------|----------|
| 5         |               |          |
| 10        |               |          |
| 20        |               |          |
| None      |               |          |

In [3]:
from sklearn.ensemble import RandomForestClassifier

results_rf = []

# Перебор max_depth при n_estimators=100
for depth in [5, 10, 20, None]:
    start = time.time()
    clf = RandomForestClassifier(n_estimators=100,
                                 max_depth=depth,
                                 random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))

    results_rf.append({'max_depth': depth,
                       'test_accuracy': round(acc, 4),
                       'time_sec': round(fit_time, 2)})
    print(f"depth={depth}: acc={acc:.4f}, time={fit_time:.2f}s")

df_rf = pd.DataFrame(results_rf)
print("\nТаблица 2. Случайный лес")
print(df_rf.to_markdown(index=False))

depth=5: acc=0.9002, time=0.29s
depth=10: acc=0.9154, time=0.46s
depth=20: acc=0.9357, time=0.79s
depth=None: acc=0.9526, time=1.47s

Таблица 2. Случайный лес
|   max_depth |   test_accuracy |   time_sec |
|------------:|----------------:|-----------:|
|           5 |          0.9002 |       0.29 |
|          10 |          0.9154 |       0.46 |
|          20 |          0.9357 |       0.79 |
|         nan |          0.9526 |       1.47 |


### 3.2. Градиентный бустинг (Gradient Boosting)

**Гиперпараметр:** `learning_rate` (фиксируем `n_estimators=100, max_depth=3`)

| learning_rate | test_accuracy | time_sec |
|---------------|---------------|----------|
| 0.01          |               |          |
| 0.05          |               |          |
| 0.1           |               |          |
| 0.5           |               |          |

In [4]:
from sklearn.ensemble import GradientBoostingClassifier

results_gb = []

# Перебор learning_rate при фиксированных n_estimators=100 и max_depth=3
for lr in [0.01, 0.05, 0.1, 0.5]:
    start = time.time()
    clf = GradientBoostingClassifier(n_estimators=100,
                                     max_depth=3,
                                     learning_rate=lr,
                                     random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_gb.append({'learning_rate': lr,
                       'test_accuracy': round(acc, 4),
                       'time_sec': round(fit_time, 2)})
    print(f"lr={lr}: acc={acc:.4f}, time={fit_time:.2f}s")

df_gb = pd.DataFrame(results_gb)
print("\nТаблица 3. Градиентный бустинг")
print(df_gb.to_markdown(index=False))

lr=0.01: acc=0.8782, time=22.80s
lr=0.05: acc=0.9475, time=22.30s
lr=0.1: acc=0.9577, time=22.76s
lr=0.5: acc=0.9662, time=22.57s

Таблица 3. Градиентный бустинг
|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|            0.01 |          0.8782 |      22.8  |
|            0.05 |          0.9475 |      22.3  |
|            0.1  |          0.9577 |      22.76 |
|            0.5  |          0.9662 |      22.57 |


### 3.3. XGBoost

**Гиперпараметр:** `gamma` (фиксируем `n_estimators=100, learning_rate=0.1`)

| gamma | test_accuracy | time_sec |
|-------|---------------|----------|
| 0     |               |          |
| 0.1   |               |          |
| 0.5   |               |          |
| 1.0   |               |          |

In [5]:
from xgboost import XGBClassifier

results_xgb = []

# Перебор gamma при фиксированных n_estimators=100, learning_rate=0.1
for g in [0, 0.1, 0.5, 1.0]:
    start = time.time()
    clf = XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        gamma=g,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_xgb.append({'gamma': g,
                        'test_accuracy': round(acc, 4),
                        'time_sec': round(fit_time, 2)})
    print(f"gamma={g}: acc={acc:.4f}, time={fit_time:.2f}s")
df_xgb = pd.DataFrame(results_xgb)
print("\nТаблица 4. XGBoost")
print(df_xgb.to_markdown(index=False))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:31:13] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


gamma=0: acc=0.9459, time=21.85s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:31:34] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


gamma=0.1: acc=0.9442, time=21.40s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:31:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


gamma=0.5: acc=0.9492, time=20.64s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:32:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


gamma=1.0: acc=0.9492, time=20.62s

Таблица 4. XGBoost
|   gamma |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|     0   |          0.9459 |      21.85 |
|     0.1 |          0.9442 |      21.4  |
|     0.5 |          0.9492 |      20.64 |
|     1   |          0.9492 |      20.62 |


### 3.4. AdaBoost

**Гиперпараметр:** `learning_rate` (фиксируем `n_estimators=100`)

| learning_rate | test_accuracy | time_sec |
|---------------|---------------|----------|
| 0.5           |               |          |
| 1.0           |               |          |
| 1.5           |               |          |
| 2.0           |               |          |

In [6]:
from sklearn.ensemble import AdaBoostClassifier

results_ada = []

# Перебор learning_rate при n_estimators=100
for lr in [0.5, 1.0, 1.5, 2.0]:
    start = time.time()
    clf = AdaBoostClassifier(n_estimators=100, learning_rate=lr, random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_ada.append({'learning_rate': lr, 'test_accuracy': round(acc, 4), 'time_sec': round(fit_time, 2)})
    print(f"lr={lr}: acc={acc:.4f}, time={fit_time:.2f}s")


df_ada = pd.DataFrame(results_ada)
print("\nТаблица 5. AdaBoost")
print(df_ada.to_markdown(index=False))

lr=0.5: acc=0.8629, time=4.21s
lr=1.0: acc=0.9205, time=3.70s
lr=1.5: acc=0.9255, time=4.22s
lr=2.0: acc=0.8731, time=3.86s

Таблица 5. AdaBoost
|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|             0.5 |          0.8629 |       4.21 |
|             1   |          0.9205 |       3.7  |
|             1.5 |          0.9255 |       4.22 |
|             2   |          0.8731 |       3.86 |


### 3.5. LightGBM

**Гиперпараметр:** `num_leaves` (фиксируем `n_estimators=100, learning_rate=0.1`)

| num_leaves | test_accuracy | time_sec |
|------------|---------------|----------|
| 15         |               |          |
| 31         |               |          |
| 63         |               |          |
| 127        |               |          |

In [7]:
from lightgbm import LGBMClassifier

results_lgbm = []

# Перебор num_leaves при фиксированных n_estimators=100, learning_rate=0.1
for leaves in [15, 31, 63, 127]:
    start = time.time()
    clf = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        num_leaves=leaves,
        random_state=42,
        n_jobs=-1,
        verbose=-1  # отключение лога итераций для чистого вывода
    )
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_lgbm.append({'num_leaves': leaves,
                         'test_accuracy': round(acc, 4),
                         'time_sec': round(fit_time, 2)})
    print(f"num_leaves={leaves}: acc={acc:.4f}, time={fit_time:.2f}s")

df_lgbm = pd.DataFrame(results_lgbm)
print("\nТаблица 6. LightGBM")
print(df_lgbm.to_markdown(index=False))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


num_leaves=15: acc=0.9662, time=2.97s


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


num_leaves=31: acc=0.9594, time=6.31s


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


num_leaves=63: acc=0.9577, time=7.46s
num_leaves=127: acc=0.9594, time=12.06s

Таблица 6. LightGBM
|   num_leaves |   test_accuracy |   time_sec |
|-------------:|----------------:|-----------:|
|           15 |          0.9662 |       2.97 |
|           31 |          0.9594 |       6.31 |
|           63 |          0.9577 |       7.46 |
|          127 |          0.9594 |      12.06 |


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### 3.6. CatBoost

**Гиперпараметр:** `depth` (фиксируем `iterations=100, learning_rate=0.1`)

| depth | test_accuracy | time_sec |
|-------|---------------|----------|
| 3     |               |          |
| 5     |               |          |
| 7     |               |          |
| 10    |               |          |

In [10]:
from catboost import CatBoostClassifier

results_cat = []

# Перебор depth при фиксированных iterations=100, learning_rate=0.1
for d in [3, 5, 7, 10]:
    start = time.time()
    clf = CatBoostClassifier(
        iterations=100,
        learning_rate=0.1,
        depth=d,
        random_state=42,
        verbose=0  # отключаем лог обучения
    )
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_cat.append({'depth': d,
                        'test_accuracy': round(acc, 4),
                        'time_sec': round(fit_time, 2)})
    print(f"depth={d}: acc={acc:.4f}, time={fit_time:.2f}s")
df_cat = pd.DataFrame(results_cat)
print("\nТаблица 7. CatBoost")
print(df_cat.to_markdown(index=False))

depth=3: acc=0.9340, time=15.63s
depth=5: acc=0.9425, time=21.46s
depth=7: acc=0.9442, time=65.67s
depth=10: acc=0.9526, time=493.53s

Таблица 7. CatBoost
|   depth |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|       3 |          0.934  |      15.63 |
|       5 |          0.9425 |      21.46 |
|       7 |          0.9442 |      65.67 |
|      10 |          0.9526 |     493.53 |


## 4. Итоговая таблица лучших результатов

In [11]:
# Автоматический поиск лучших значений во всех таблицах
summary_data = []

# Название модели, DataFrame, Имя столбца с тестируемым гиперпараметром
models_list = [
    ('Decision Tree', df_tree, 'max_depth'),
    ('Random Forest', df_rf, 'max_depth'),
    ('Gradient Boosting', df_gb, 'learning_rate'),
    ('XGBoost', df_xgb, 'gamma'),
    ('AdaBoost', df_ada, 'learning_rate'),
    ('LightGBM', df_lgbm, 'num_leaves'),
    ('CatBoost', df_cat, 'depth')
]

for name, df, param_col in models_list:
    # Находим строку с максимальной test_accuracy
    best_idx = df['test_accuracy'].idxmax()
    best_row = df.loc[best_idx]

    summary_data.append({
        'Algorithm': name,
        'Best params': f"{param_col}={best_row[param_col]}",
        'Best test acc': best_row['test_accuracy'],
        'Time (sec)': best_row['time_sec']
    })

summary = pd.DataFrame(summary_data)

# сортировка по точности по убыванию
summary = summary.sort_values('Best test acc', ascending=False).reset_index(drop=True)

print("Итоговая таблица лучших результатов")
print(summary.to_markdown(index=False))

Итоговая таблица лучших результатов
| Algorithm         | Best params       |   Best test acc |   Time (sec) |
|:------------------|:------------------|----------------:|-------------:|
| Gradient Boosting | learning_rate=0.5 |          0.9662 |        22.57 |
| LightGBM          | num_leaves=15.0   |          0.9662 |         2.97 |
| Random Forest     | max_depth=nan     |          0.9526 |         1.47 |
| CatBoost          | depth=10.0        |          0.9526 |       493.53 |
| XGBoost           | gamma=0.5         |          0.9492 |        20.64 |
| AdaBoost          | learning_rate=1.5 |          0.9255 |         4.22 |
| Decision Tree     | max_depth=nan     |          0.8697 |         0.4  |


## 5. Вопросы (ответить текстом в следующей ячейке)

1. Какой алгоритм показал максимальную точность? Какие параметры к этому привели?

2. Какой алгоритм быстрее всего обучался? Во сколько раз он быстрее самого медленного?

3. У каких алгоритмов наблюдалось переобучение? При каких параметрах?

4. Какой алгоритм вы выбрали бы для продакшн-системы с миллионом текстов? Почему?

# Напишите свои ответы здесь

**Ответ 1:**

Максимальную точность (0.9662) показали: **Gradient Boosting с learning_rate=0.5** и **LightGBM с num_leaves=15.0**

**Ответ 2:**

Самый быстрый это Decision Tree: 0.4 сек, самый медленный CatBoost: 493.53 сек.
Разница более, чем в 1000 раз.

**Ответ 3:**

Переобучение наблюдалось у моделей, где качество на обучающей выборке значительно превышало качество на тестовой. Cлишком сложные модели начинают запоминать обучающие данные вместо выявления общих закономерностей.

Наиболее вероятно переобучались:

Decision Tree при max_depth = None (дерево росло без ограничений глубины)
Random Forest при max_depth = None
CatBoost при depth = 10
XGBoost при высоких значениях сложности модели, например gamma = 0.5

**Ответ 4:**

Для продакшн-системы с большим объёмом данных я бы выбрала LightGBM. Хотя Decision Tree и Random Forest обучаются быстро, их точность заметно ниже. CatBoost показал хорошие результаты, но время обучения слишком велико для системы с миллионом текстов. LightGBM даёт лучший баланс между качеством и производительностью:

- максимальная точность (0.9662),
- высокая скорость обучения (2.97 сек),
- хорошая масштабируемость,
- низкое потребление памяти,
- эффективная работа на больших датасетах и признаках, характерных для текстов.



## Критерии оценки

| Что оценивается | Баллы |
|----------------|-------|
| Заполнены все 7 таблиц (правильно собраны accuracy и время) | 3 |
| Заполнена итоговая таблица лучших результатов | 1 |
| Ответы на 4 вопроса (по 0.5 балла) | 2 |
| Код воспроизводим (random_state=42, порядок ячеек корректен) | 1 |
| **Качество кода** (отсутствие дублирования, осмысленные имена переменных, циклы вместо копипасты, использование .to_markdown()) | 3 |
| **Итого** | **10** |

### Детали по качеству кода (+3 балла):
- **+1** — использование единого шаблона для всех экспериментов (цикл + list.append + pd.DataFrame)
- **+1** — правильное форматирование вывода (округление времени и accuracy до 2-4 знаков)
- **+1** — читаемые имена переменных, комментарии, отсутствие `eval()`, `exec()` и магических чисел

**Штрафы:**
- -1 балл, если код не запускается без ошибок
- -1 балл, если отсутствует `random_state=42` в любой из моделей
- -1 балл, если таблицы выведены криво (не через `to_markdown` или нечитаемый `print`)